Load all 5 cities

In [22]:
import pandas as pd
import numpy as np

BASE = "C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data"

cities = {
    'MY1':  'London',
    'BIRR': 'Birmingham',
    'MAN3': 'Manchester',
    'NEWC': 'Newcastle',
    'CARD': 'Cardiff',
}

dfs = []
for site_id, name in cities.items():
    # Load 2021-2024
    df_train = pd.read_csv(f"{BASE}/{name}/merged/{site_id}_merged.csv",
                           parse_dates=['datetime'])
    # Load 2025
    df_test = pd.read_csv(f"{BASE}/test/merged/{site_id}_merged_2025.csv",
                          parse_dates=['datetime'])
    dfs.append(df_train)
    dfs.append(df_test)

data = pd.concat(dfs, ignore_index=True)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

# Remove the duplicate boundary row (2025-01-01 00:00 appears in both files)
data = data.drop_duplicates(subset=['city','datetime'])
data = data.reset_index(drop=True)

print(f"Total rows: {len(data):,}")
print(f"Date range: {data['datetime'].min()} → {data['datetime'].max()}")
print(data.groupby('city').size())

Total rows: 219,120
Date range: 2021-01-01 01:00:00 → 2026-01-01 00:00:00
city
BIRR    43824
CARD    43824
MAN3    43824
MY1     43824
NEWC    43824
dtype: int64


Clip negative values

In [23]:
# Instrument noise near zero produces small negatives — clip to 0
for col in ['NO2','PM2.5','PM10','O3','SO2']:
    neg_count = (data[col] < 0).sum()
    if neg_count > 0:
        print(f"{col}: {neg_count} negative values clipped to 0")
    data[col] = data[col].clip(lower=0)

print("\nMin values after clipping:")
print(data[['NO2','PM2.5','PM10','O3','SO2']].min())

NO2: 1 negative values clipped to 0
PM2.5: 1321 negative values clipped to 0
PM10: 655 negative values clipped to 0
O3: 33 negative values clipped to 0
SO2: 170 negative values clipped to 0

Min values after clipping:
NO2      0.0
PM2.5    0.0
PM10     0.0
O3       0.0
SO2      0.0
dtype: float64


Impute missing values

In [24]:
# Strategy:
# Gap <= 3 hours → forward fill (instrument blip)
# Gap > 3 hours  → fill with column median per city (longer outage)

def impute_city(df):
    df = df.copy().sort_values('datetime')
    cols = ['NO2','PM2.5','PM10','O3','SO2','temp','humidity','wind_speed','pressure']
    
    for col in cols:
        # Forward fill short gaps (up to 3 consecutive hours)
        df[col] = df[col].fillna(method='ffill', limit=3)
        # Fill remaining with city median
        df[col] = df[col].fillna(df[col].median())
    
    return df

data = data.groupby('city', group_keys=False).apply(impute_city)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

print("Missing values after imputation:")
print(data[['NO2','PM2.5','PM10','O3','SO2','temp','humidity','wind_speed','pressure']].isna().sum())

Missing values after imputation:
NO2               0
PM2.5             0
PM10              0
O3                0
SO2           87648
temp              0
humidity          0
wind_speed        0
pressure          0
dtype: int64


C:\Users\nikes\AppData\Local\Temp\ipykernel_24332\1671484326.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill', limit=3)
C:\Users\nikes\AppData\Local\Temp\ipykernel_24332\1671484326.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill', limit=3)
C:\Users\nikes\AppData\Local\Temp\ipykernel_24332\1671484326.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill', limit=3)
C:\Users\nikes\AppData\Local\Temp\ipykernel_24332\1671484326.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill

Compute DAQI label

In [25]:
# Official DEFRA DAQI thresholds
def compute_daqi(row):
    def sub_index(val, bands):
        if pd.isna(val):
            return 1
        for i, threshold in enumerate(bands):
            if val <= threshold:
                return i + 1
        return 10

    no2_bands  = [67, 134, 200, 267, 334, 400, 467, 534, 600]
    pm25_bands = [11,  23,  35,  41,  47,  53,  58,  64,  70]
    pm10_bands = [16,  33,  50,  58,  66,  75,  83,  91, 100]
    o3_bands   = [33,  66, 100, 120, 140, 160, 187, 213, 240]
    so2_bands  = [88, 177, 266, 354, 443, 532, 710, 887, 1064]

    indices = [
        sub_index(row['NO2'],   no2_bands),
        sub_index(row['PM2.5'], pm25_bands),
        sub_index(row['PM10'],  pm10_bands),
        sub_index(row['O3'],    o3_bands),
        sub_index(row['SO2'],   so2_bands),
    ]
    return max(indices)

print("Computing DAQI scores... (takes ~1 minute)")
data['DAQI_score'] = data.apply(compute_daqi, axis=1)

# Map score to band label
def daqi_band(score):
    if score <= 3:  return 'Low'
    if score <= 6:  return 'Moderate'
    if score <= 9:  return 'High'
    return 'Very High'

data['DAQI_label'] = data['DAQI_score'].apply(daqi_band)

print("\nDAQI distribution across all cities:")
print(data['DAQI_label'].value_counts())
print("\nDAQI by city:")
print(data.groupby('city')['DAQI_label'].value_counts().unstack(fill_value=0))

Computing DAQI scores... (takes ~1 minute)

DAQI distribution across all cities:
DAQI_label
Low          214422
Moderate       4167
High            433
Very High        98
Name: count, dtype: int64

DAQI by city:
DAQI_label  High    Low  Moderate  Very High
city                                        
BIRR         108  42675      1027         14
CARD          49  42834       928         13
MAN3          53  43044       705         22
MY1          184  42565      1029         46
NEWC          39  43304       478          3


Extract time features

In [26]:
data['hour']       = data['datetime'].dt.hour
data['day_of_week'] = data['datetime'].dt.dayofweek   # 0=Monday, 6=Sunday
data['month']      = data['datetime'].dt.month
data['year']       = data['datetime'].dt.year

# Season: 0=Winter, 1=Spring, 2=Summer, 3=Autumn
def get_season(month):
    if month in [12, 1, 2]:  return 0  # Winter
    if month in [3, 4, 5]:   return 1  # Spring
    if month in [6, 7, 8]:   return 2  # Summer
    return 3                            # Autumn

data['season'] = data['month'].apply(get_season)

# Weekend flag
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)

print("New time feature columns added:")
print(data[['datetime','hour','day_of_week','month','year','season','is_weekend']].head(5))

New time feature columns added:
             datetime  hour  day_of_week  month  year  season  is_weekend
0 2021-01-01 01:00:00     1            4      1  2021       0           0
1 2021-01-01 02:00:00     2            4      1  2021       0           0
2 2021-01-01 03:00:00     3            4      1  2021       0           0
3 2021-01-01 04:00:00     4            4      1  2021       0           0
4 2021-01-01 05:00:00     5            4      1  2021       0           0


Encode city as a number

In [27]:
# XGBoost needs numbers not strings
city_map = {'MY1': 0, 'BIRR': 1, 'MAN3': 2, 'NEWC': 3, 'CARD': 4}
data['city_code'] = data['city'].map(city_map)

print("City encoding:")
print(data[['city','city_code']].drop_duplicates().sort_values('city_code'))

City encoding:
        city  city_code
131472   MY1          0
0       BIRR          1
87648   MAN3          2
175296  NEWC          3
43824   CARD          4


Encode DAQI label as a number

In [28]:
# 0=Low, 1=Moderate, 2=High, 3=Very High
label_map = {'Low': 0, 'Moderate': 1, 'High': 2, 'Very High': 3}
data['DAQI_class'] = data['DAQI_label'].map(label_map)

print("Class distribution:")
print(data['DAQI_class'].value_counts().sort_index())
print("\nClass balance (%):")
print((data['DAQI_class'].value_counts(normalize=True) * 100).round(1).sort_index())

Class distribution:
DAQI_class
0    214422
1      4167
2       433
3        98
Name: count, dtype: int64

Class balance (%):
DAQI_class
0    97.9
1     1.9
2     0.2
3     0.0
Name: proportion, dtype: float64


Train/test split

In [29]:
# Chronological split — never random on time series
# Train: 2021-2024  |  Test: 2025

train = data[data['year'] <= 2024].copy()
test  = data[data['year'] == 2025].copy()

print(f"Train: {len(train):,} rows  ({train['datetime'].min().date()} → {train['datetime'].max().date()})")
print(f"Test:  {len(test):,} rows   ({test['datetime'].min().date()} → {test['datetime'].max().date()})")
print(f"\nTrain class balance:")
print((train['DAQI_class'].value_counts(normalize=True) * 100).round(1).sort_index())
print(f"\nTest class balance:")
print((test['DAQI_class'].value_counts(normalize=True) * 100).round(1).sort_index())

Train: 175,315 rows  (2021-01-01 → 2024-12-31)
Test:  43,800 rows   (2025-01-01 → 2025-12-31)

Train class balance:
DAQI_class
0    98.0
1     1.7
2     0.2
3     0.0
Name: proportion, dtype: float64

Test class balance:
DAQI_class
0    97.1
1     2.6
2     0.2
3     0.1
Name: proportion, dtype: float64


Save final datasets

In [30]:
import os
os.makedirs(f"{BASE}/../models", exist_ok=True)

# Save full engineered dataset
data.to_csv(f"{BASE}/../models/full_dataset.csv", index=False)

# Save train and test splits
train.to_csv(f"{BASE}/../models/train.csv", index=False)
test.to_csv(f"{BASE}/../models/test.csv",  index=False)

print(f"Saved:")
print(f"  models/full_dataset.csv  — {len(data):,} rows")
print(f"  models/train.csv         — {len(train):,} rows")
print(f"  models/test.csv          — {len(test):,} rows")

print(f"\nFinal feature columns:")
feature_cols = ['city_code','hour','day_of_week','month','season','is_weekend',
                'NO2','PM2.5','PM10','O3','SO2','temp','humidity','wind_speed','pressure']
print(feature_cols)
print(f"\nTarget column: DAQI_class (0=Low, 1=Moderate, 2=High, 3=Very High)")

Saved:
  models/full_dataset.csv  — 219,120 rows
  models/train.csv         — 175,315 rows
  models/test.csv          — 43,800 rows

Final feature columns:
['city_code', 'hour', 'day_of_week', 'month', 'season', 'is_weekend', 'NO2', 'PM2.5', 'PM10', 'O3', 'SO2', 'temp', 'humidity', 'wind_speed', 'pressure']

Target column: DAQI_class (0=Low, 1=Moderate, 2=High, 3=Very High)
